# 🧠 traceback-coach — memory tour

The coach can **remember which Python errors you keep struggling with**, across
kernel restarts, and give you a personalized Socratic review on demand.

It's **optional and offline-first**:
- Install the extra — `pip install "traceback-coach[hermes]"` — and memory turns on,
  backed by an isolated **Hermes profile** (its own store, no API key handled by the app).
- Without it, the coach behaves exactly as before: no memory, no provider calls.

This notebook runs **fully offline (zero tokens)**. It drives the memory *engine*
directly so you can watch it work, then shows the live `%coach_*` magics for a
Hermes-equipped environment.

## 1 · The memory engine — live, offline

In real use the store lives inside the coach's Hermes profile. For this demo we
point it at a throwaway file and simulate a learner across several days.

In [1]:
import json, os, tempfile
from traceback_coach.hermes_memory import HermesMemory

store = os.path.join(tempfile.mkdtemp(), "traceback_coach_history.md")
mem = HermesMemory(store_path=store)

# The learner keeps hitting IndexError; NameError shows up once; they fix an IndexError.
for day in ("2026-07-10", "2026-07-11", "2026-07-12", "2026-07-14"):
    mem.record("IndexError", day)
mem.record("NameError", "2026-07-12")
mem.record_fixed("IndexError")

print("summary       :", json.dumps(mem.summary(), indent=2))
print("top weakness  :", mem.top_weakness())

summary       : {
  "IndexError": {
    "seen": 4,
    "fixed": 1,
    "last": "2026-07-14"
  },
  "NameError": {
    "seen": 1,
    "fixed": 0,
    "last": "2026-07-12"
  }
}
top weakness  : ('IndexError', 4)


### The entire store (this is all that's persisted)
Family names, counts, and dates — **never your code, variables, or messages.**

In [2]:
print(open(store).read())

# Python error history (maintained by traceback-coach)

Each line: <ErrorFamily>: seen <N>, fixed <M>, last <YYYY-MM-DD>

- IndexError: seen 4, fixed 1, last 2026-07-14
- NameError: seen 1, fixed 0, last 2026-07-12



## 2 · The guiding question personalizes on chronic errors — offline

The coach never hands you the fix. When an error becomes a *pattern*, the
deterministic question gains a chronic nudge — no LLM required.

In [3]:
from traceback_coach._core import make_question, parse_traceback
from traceback_coach.knowledge import lookup

def parsed_from(src):
    try:
        exec(src, {})
    except Exception as e:
        return parse_traceback(type(e), e, e.__traceback__, src)

parsed = parsed_from("[][5]")
fam = lookup("IndexError")
off = lambda *a: ""   # force the offline template (no LLM)

print("1st time :", make_question(parsed, fam, cell_source="[][5]", llm=off, seen_count=1))
print()
print("9th time :", make_question(parsed, fam, cell_source="[][5]", llm=off, seen_count=9))

1st time : How long is the sequence, and what is the largest valid index?

9th time : How long is the sequence, and what is the largest valid index? You've hit IndexError 9 times now — what's your rule for avoiding it?


## 3 · `%coach_insights` — the Socratic review (agentic)

On demand, the coach hands your history to a Hermes agent and asks for a Socratic
review — the **one** place a provider is ever used. Here is the exact prompt it
builds, then an illustrative result (the agent is stubbed so this stays offline).

In [4]:
print(mem._build_reflect_prompt(lang="en"))

You are a Socratic Python coach. Here is a learner's error history (most frequent first):
- IndexError: seen 4 times, fixed 1
- NameError: seen 1 times, fixed 0

In 3-4 sentences, name their biggest weakness and ask ONE guiding question that helps them self-correct next time. Do NOT give the fix or any code. Reply in English.


In [5]:
# Stub the agent so this cell runs with no Hermes. With a live Hermes, reflect()
# sends the prompt above and returns the agent's REAL Socratic review.
import traceback_coach.hermes_memory as hm

async def _demo_agent(prompt, profile, command):
    return ("Your recurring weakness is IndexError — 4 hits, 1 fixed. Before you "
            "index a sequence, ask yourself: for a list of length n, what is the "
            "largest valid index, and how can you check you're within it?")

hm._drive_agent = _demo_agent
print(mem.reflect(lang="en"))

Your recurring weakness is IndexError — 4 hits, 1 fixed. Before you index a sequence, ask yourself: for a list of length n, what is the largest valid index, and how can you check you're within it?


## 4 · The real thing — `%coach_*` magics in a Hermes-equipped environment

Everything above used the engine directly. In a kernel with
`traceback-coach[hermes]` installed and a **Hermes ≥ 0.18** profile available
(e.g. a JupyterHub where Hermes is preconfigured), it's all automatic:

```python
%load_ext traceback_coach
%coach_memory status     # is memory on? where's the store?
%coach_memory on         # (auto-on when the [hermes] extra is installed)
%coach_watch on

# ... write code, make mistakes; the coach explains each error with a guiding
#     question and quietly remembers which families you repeat ...

%coach_stats             # your error history, across sessions
%coach_insights          # a Socratic review of your weaknesses (uses the agent once)
%coach_forget            # erase the saved history
```

- **No key, no Hermes** → memory stays off and the coach works exactly as before.
- Only `%coach_insights` ever calls a provider, and only when *you* ask.
- The store lives in the coach's own isolated Hermes profile — the provider is
  cloned from the host, so **the coach never handles an API key**.